In [139]:
import pandas
import pathlib
import numpy as np
import json
from collections import OrderedDict
from datasets.utils import BASE_DIR

In [140]:
file = '2025.json'

In [141]:
with open(BASE_DIR.joinpath('wnba', file), mode='r') as f:
    data = json.load(f)

In [142]:
columns = list(map(lambda x: x['name'], data['columns']))

In [143]:
from typing import Callable, Generator, Tuple, List


T = Generator[List[Tuple[str, int]]]

R = Generator[OrderedDict[str, int]]


def create_data(func: Callable[[], T]) -> R:
    def wrapper():
        results = func()
        
        for result in results:
            yield OrderedDict(result)
    return wrapper

@create_data
def zip_columns() -> T:
    for items in data['data']:
        yield list(zip(columns, items))


result = zip_columns()

In [144]:
column_dtypes = {column: np.float32 for column in columns if column not in ['PLAYER', 'ID', 'TEAM']}

In [145]:
df = pandas.DataFrame(result)

In [146]:
df = df.astype(column_dtypes)

In [147]:
df = df.drop(columns=['ID'])

In [148]:
df = df.sort_values('MIN', ascending=False)

In [149]:
def round_value(value: float):
    return round(value, 1)

In [150]:
def calc_ppg(df: pandas.DataFrame):
    df['PPG'] = (df['PTS'] / df['GP'])
    df['PPG'] = df['PPG'].map(round_value)
    return  df

def calc_rpg(df: pandas.DataFrame):
    df['RPG'] = df['REB'] / df['GP']
    df['RPG'] = df['RPG'].map(round_value)
    return df

def calc_apg(df: pandas.DataFrame):
    df['APG'] = df['AST'] / df['GP']
    df['APG'] = df['APG'].map(round_value)
    return df

df = df.pipe(calc_ppg).pipe(calc_rpg).pipe(calc_apg)

In [151]:
df.get(['PLAYER', 'PPG'])

,PLAYER,PPG
7,Arike Ogunbowale,16.8
10,Rhyne Howard,16.8
0,Allisha Gray,19.5
2,Kelsey Plum,20.6
23,Gabby Williams,13.4
...,...,...
164,Moriah Jefferson,0.0
154,Bree Hall,1.5
157,Aerial Powers,1.0
159,Kiana Williams,2.0


In [152]:
df.to_csv('stats.csv', index_label='id')